# musubi-tuner — FLUX.2\[klein\] 9B LoRA Training

Train a LoRA for **FLUX.2-klein-9B** (Black Forest Labs) using [musubi-tuner](https://github.com/kohya-ss/musubi-tuner) by kohya-ss.

FLUX.2-klein-9B is an **image generation / image editing** model.

---

## ⚠️ Prerequisites — do these BEFORE running any cells

### 1. Accept model licenses on Hugging Face
You must accept the license for **both** gated repos:
- [black-forest-labs/FLUX.2-klein-9B](https://huggingface.co/black-forest-labs/FLUX.2-klein-9B) — DiT + Text Encoder
- [black-forest-labs/FLUX.2-dev](https://huggingface.co/black-forest-labs/FLUX.2-dev) — AE (autoencoder)

### 2. Create a Hugging Face token
Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → New token → **Read** access.

### 3. Add the token to Colab Secrets
In Colab: **🔑 Secrets** (left sidebar) → Add new secret → Name: `HF_TOKEN`, Value: `hf_...`

### 4. Runtime → Change runtime type → **A100 GPU**
The 9B model requires ~40 GB VRAM. A100 (40 GB) is the minimum recommended GPU.
A T4 (16 GB) may work with maximum memory optimizations but is not guaranteed.

---

## Training note: distilled vs. base

FLUX.2-klein-9B is a **distilled** model optimized for fast inference (4 steps).  
For best LoRA training results, the musubi-tuner docs recommend **FLUX.2-klein-base-9B** instead.  
To switch, change `MODEL_VERSION` in Section 6 to `klein-base-9b` and update the DiT filename.
This notebook defaults to `klein-9b` as requested.


## Section 1 — GPU Check


In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    vram = p.total_memory / 1024**3
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {vram:.1f} GB")
    if vram < 30:
        print("\n⚠️  < 30 GB VRAM — enable ALL memory-saving flags in Section 6.")
        print("   Set BLOCKS_TO_SWAP=16, USE_FP8=True, GRADIENT_CHECKPOINTING_CPU_OFFLOAD=True")
    else:
        print("\n✅ Sufficient VRAM for FLUX.2-klein-9B training.")
else:
    print("\n❌ No GPU — change runtime type before continuing.")


## Section 2 — Mount Google Drive

All model weights (~60 GB total) and outputs are stored on Google Drive to persist across sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted.")


## Section 3 — Install musubi-tuner

Safe to skip if `/content/musubi-tuner` already exists from a previous session.


In [ ]:
import os

REPO_DIR = "/content/musubi-tuner"

if not os.path.exists(REPO_DIR):
    print("Cloning musubi-tuner...")
    !git clone https://github.com/kohya-ss/musubi-tuner {REPO_DIR} -q
else:
    print("Repo exists — pulling latest...")
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}

import torch
ver = tuple(int(x) for x in torch.__version__.split('.')[:2])
if ver < (2, 5):
    print("Upgrading PyTorch to 2.5.1+...")
    !pip install -q torch==2.5.1 torchvision --index-url https://download.pytorch.org/whl/cu124
else:
    print(f"PyTorch {torch.__version__} — OK")

print("Installing musubi-tuner + extras...")
!pip install -q -e .
!pip install -q bitsandbytes tensorboard

print("\n✅ Done.")


## Section 4 — Hugging Face Authentication

Reads your `HF_TOKEN` from Colab Secrets and logs in via the `hf` CLI.
The token is stored only for this session and never written to disk.


In [ ]:
import os

# Try Colab Secrets first, then fall back to manual input
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN secret is empty")
    print("✅ Token loaded from Colab Secrets.")
except Exception as e:
    print(f"Secrets unavailable ({e}) — falling back to manual input.")
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HuggingFace token (hf_...): ")

# Authenticate via hf CLI
!hf auth login --token {HF_TOKEN}

# Also expose as env var for subprocesses
os.environ['HF_TOKEN'] = HF_TOKEN

print("\n✅ Authenticated with Hugging Face Hub.")
print("   Verify you have accepted licenses for:")
print("   • https://huggingface.co/black-forest-labs/FLUX.2-klein-9B")
print("   • https://huggingface.co/black-forest-labs/FLUX.2-dev")


## Section 5 — Create Directories


In [ ]:
import os

DRIVE_ROOT   = "/content/drive/MyDrive/musubi-tuner-flux2"
MODEL_DIR    = f"{DRIVE_ROOT}/models"
TE_DIR       = f"{MODEL_DIR}/flux2_klein_9b_te"   # text encoder shards
DATASET_DIR  = f"{DRIVE_ROOT}/dataset"              # your training images + captions
CACHE_DIR    = f"{DRIVE_ROOT}/cache"
OUTPUT_DIR   = f"{DRIVE_ROOT}/outputs"

for d in [MODEL_DIR, TE_DIR, DATASET_DIR, CACHE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✅ {d}")


## Section 6 — Configuration

**Edit this cell** before proceeding. All later cells read from these variables.


In [ ]:
# ============================================================
#   MODEL
# ============================================================
MODEL_VERSION = "klein-9b"          # klein-9b (distilled) or klein-base-9b (better for training)
DIT_FILE      = "flux2-klein-9b.safetensors"     # DiT weights filename in the repo
AE_FILE       = "ae.safetensors"                  # autoencoder (from FLUX.2-dev repo)
LORA_NAME     = "flux2-klein-9b-lora"             # output name (no extension)

# ============================================================
#   DATASET
# ============================================================
# Put your images + .txt captions in DATASET_DIR:
#   image001.jpg  +  image001.txt  (caption: "a photo of sks person...")
#   image002.png  +  image002.txt
#   ...
#
# For image-editing LoRA (reference-based), also add a control_directory
# with matching reference images — see Section 7 for details.

CAPTION_EXT         = ".txt"
RESOLUTION          = [1024, 1024]  # [W, H] — FLUX.2 is optimized for 1024x1024
BATCH_SIZE          = 1
ENABLE_BUCKET       = True
NUM_REPEATS         = 1

# Set to a path if doing image-editing LoRA (reference images).
# Set to None for style/character LoRA without reference images.
CONTROL_DIR         = None          # e.g., f"{DRIVE_ROOT}/control_images"

# ============================================================
#   LoRA PARAMETERS
# ============================================================
NETWORK_DIM         = 32            # LoRA rank (16-64)
NETWORK_ALPHA       = 16            # usually NETWORK_DIM / 2

# ============================================================
#   TRAINING
# ============================================================
LEARNING_RATE       = 1e-4
MAX_TRAIN_EPOCHS    = 16
SAVE_EVERY_N_EPOCHS = 1
SEED                = 42
OPTIMIZER_TYPE      = "adamw8bit"
MIXED_PRECISION     = "bf16"

# ============================================================
#   MEMORY OPTIMIZATION
# ============================================================
USE_FP8                          = True   # --fp8_base --fp8_scaled (recommended for 9B)
USE_FP8_TEXT_ENCODER             = True   # --fp8_text_encoder (Qwen3 supports this)
VAE_DTYPE                        = "bfloat16"  # reduces VRAM; use "float32" if quality issues
GRADIENT_CHECKPOINTING           = True
GRADIENT_CHECKPOINTING_CPU_OFFLOAD = False  # set True only if still OOM
BLOCKS_TO_SWAP                   = 0      # max=16 for klein-9b with fp8+fp8_scaled; increase if OOM

# ============================================================
#   LOGGING
# ============================================================
USE_TENSORBOARD = False

print("✅ Config loaded.")
print(f"   Model version : {MODEL_VERSION}")
print(f"   Resolution    : {RESOLUTION[0]}x{RESOLUTION[1]}")
print(f"   LoRA dim      : {NETWORK_DIM}")
print(f"   Epochs        : {MAX_TRAIN_EPOCHS}")
print(f"   FP8 DiT       : {USE_FP8}")
print(f"   FP8 TextEnc   : {USE_FP8_TEXT_ENCODER}")
print(f"   Blocks swap   : {BLOCKS_TO_SWAP}")


## Section 7 — Download Models

Uses the `hf` CLI to download from gated Hugging Face repos.
Files are saved to Google Drive and skipped on subsequent runs.

### Files downloaded
| File | Repo | Size |
|---|---|---|
| `flux2-klein-9b.safetensors` | FLUX.2-klein-9B | ~18 GB |
| Text encoder shards (×4) | FLUX.2-klein-9B | ~16 GB |
| `ae.safetensors` | FLUX.2-dev | ~0.3 GB |
| **Total** | | **~34 GB** |

Downloads to Google Drive take 15–40 minutes depending on connection speed.


In [ ]:
import os, glob

def hf_download(repo_id, filename=None, local_dir=None,
                include=None, exclude=None, skip_if_exists=True):
    """Thin wrapper around `hf download` CLI."""
    if filename and skip_if_exists:
        dest = os.path.join(local_dir, filename)
        if os.path.exists(dest):
            print(f"  ⏭️  {filename} — already exists, skipping")
            return dest

    parts = ["hf", "download", repo_id]
    if filename:
        parts.append(filename)
    if local_dir:
        parts += ["--local-dir", local_dir]
    if include:
        parts += ["--include", include]
    if exclude:
        parts += ["--exclude", exclude]
    parts += ["--token", HF_TOKEN]

    cmd = " ".join(f'"{p}"' if " " in p else p for p in parts)
    print(f"  ⬇️  {cmd}")
    os.system(cmd)


# ---- 1. DiT weights ----
print("=== 1/3 DiT weights ===")
hf_download(
    repo_id="black-forest-labs/FLUX.2-klein-9B",
    filename=DIT_FILE,
    local_dir=MODEL_DIR,
)

# ---- 2. Text encoder shards (Qwen3 8B, 4 shards) ----
print("\n=== 2/3 Text encoder shards ===")
existing_shards = glob.glob(f"{TE_DIR}/*-of-*.safetensors")
if len(existing_shards) >= 4:
    print(f"  ⏭️  {len(existing_shards)} shard files found — skipping download")
else:
    hf_download(
        repo_id="black-forest-labs/FLUX.2-klein-9B",
        local_dir=TE_DIR,
        include="*-of-*.safetensors",   # shard pattern: 00001-of-00004.safetensors
    )

# ---- 3. AE (autoencoder / VAE) from FLUX.2-dev ----
print("\n=== 3/3 Autoencoder (ae.safetensors) ===")
hf_download(
    repo_id="black-forest-labs/FLUX.2-dev",
    filename=AE_FILE,
    local_dir=MODEL_DIR,
)

# ---- Resolve text encoder first-shard path ----
te_shards = sorted(glob.glob(f"{TE_DIR}/*-of-*.safetensors"))
if te_shards:
    TEXT_ENCODER_PATH = te_shards[0]
    print(f"\n✅ Text encoder first shard: {TEXT_ENCODER_PATH}")
    print(f"   Total shards: {len(te_shards)}")
else:
    print("\n⚠️  No text encoder shards found — re-run download or check TE_DIR.")
    TEXT_ENCODER_PATH = None

DIT_PATH = os.path.join(MODEL_DIR, DIT_FILE)
AE_PATH  = os.path.join(MODEL_DIR, AE_FILE)

print(f"   DiT  : {DIT_PATH}  (exists={os.path.exists(DIT_PATH)})")
print(f"   AE   : {AE_PATH}  (exists={os.path.exists(AE_PATH)})")


## Section 8 — Prepare Dataset

### Upload your images to Google Drive

Place images + `.txt` captions in the `DATASET_DIR` folder:
```
dataset/
├── img001.jpg
├── img001.txt      ← "a photo of sks person smiling"
├── img002.png
├── img002.txt
└── ...
```

### For image-editing LoRA (optional)
If you want to train a reference-image editing LoRA (like inpainting or style transfer), also create a `control_images/` folder with matching reference images and set `CONTROL_DIR` in Section 6.

### Caption tips
- Use a unique trigger word in every caption: `sks`, `ohwx`, `TOK`
- Be descriptive: `"a photo of ohwx man in a blue jacket walking outdoors"`
- 15–50 images is enough for most LoRA use cases

Run the cell below to generate the dataset TOML config.


In [ ]:
import os

try:
    import toml
except ImportError:
    !pip install -q toml
    import toml

TOML_PATH = f"{DRIVE_ROOT}/dataset_config.toml"

ds_entry = {
    "image_directory": DATASET_DIR,
    "cache_directory": CACHE_DIR,
    "num_repeats": NUM_REPEATS,
}
if CONTROL_DIR is not None:
    ds_entry["control_directory"] = CONTROL_DIR

config = {
    "general": {
        "resolution": RESOLUTION,
        "caption_extension": CAPTION_EXT,
        "batch_size": BATCH_SIZE,
        "enable_bucket": ENABLE_BUCKET,
        "bucket_no_upscale": False,
    },
    "datasets": [ds_entry],
}

with open(TOML_PATH, "w") as f:
    toml.dump(config, f)

print(f"✅ Dataset config → {TOML_PATH}")
print()
print(open(TOML_PATH).read())

# Count files
if os.path.exists(DATASET_DIR):
    imgs = [f for f in os.listdir(DATASET_DIR)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    caps = [f for f in os.listdir(DATASET_DIR) if f.endswith(CAPTION_EXT)]
    print(f"Dataset: {len(imgs)} images, {len(caps)} captions")
    if not imgs:
        print("⚠️  No images found — upload training data to DATASET_DIR before caching.")
    elif len(caps) < len(imgs):
        print(f"⚠️  {len(imgs) - len(caps)} images are missing caption files.")
else:
    print(f"⚠️  DATASET_DIR not found: {DATASET_DIR}")


## Section 9 — Configure Accelerate


In [ ]:
import os, yaml

accel_cfg = {
    "compute_environment": "LOCAL_MACHINE",
    "debug": False,
    "distributed_type": "NO",
    "downcast_bf16": "no",
    "enable_cpu_affinity": False,
    "gpu_ids": "0",
    "machine_rank": 0,
    "main_training_function": "main",
    "mixed_precision": MIXED_PRECISION,
    "num_machines": 1,
    "num_processes": 1,
    "rdzv_backend": "static",
    "same_network": True,
    "tpu_env": [],
    "tpu_use_cluster": False,
    "tpu_use_sudo": False,
    "use_cpu": False,
}

accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
accel_path = f"{accel_dir}/default_config.yaml"

with open(accel_path, "w") as f:
    yaml.dump(accel_cfg, f)

print(f"✅ Accelerate config written ({MIXED_PRECISION} mixed precision).")


## Section 10 — Cache Latents

Encodes your training images through the autoencoder and saves the latents.  
Runs once; results are reused for all training epochs.


In [ ]:
import os
REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

vae_dtype_flag = f"--vae_dtype {VAE_DTYPE}" if VAE_DTYPE != "float32" else ""

cmd = (
    f"python src/musubi_tuner/flux_2_cache_latents.py"
    f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
    f" --vae {AE_PATH}"
    f" --model_version {MODEL_VERSION}"
    f" {vae_dtype_flag}"
)

print("Latent caching command:")
print(cmd)
print()
!{cmd}


## Section 11 — Cache Text Encoder Outputs

Encodes all captions through the Qwen3 8B text encoder.  
The text encoder (~16 GB) is only loaded here — not during training.


In [ ]:
import os
REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

if TEXT_ENCODER_PATH is None:
    raise RuntimeError("TEXT_ENCODER_PATH not set — re-run Section 7 to download text encoder shards.")

fp8_te_flag = "--fp8_text_encoder" if USE_FP8_TEXT_ENCODER else ""

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
batch = 4 if vram >= 30 else 1

cmd = (
    f"python src/musubi_tuner/flux_2_cache_text_encoder_outputs.py"
    f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
    f" --text_encoder {TEXT_ENCODER_PATH}"
    f" --model_version {MODEL_VERSION}"
    f" --batch_size {batch}"
    f" {fp8_te_flag}"
)

print("Text encoder caching command:")
print(cmd)
print()
!{cmd}


## Section 12 — Train LoRA

Checkpoints saved to `OUTPUT_DIR` every `SAVE_EVERY_N_EPOCHS` epochs.

### Memory reference for FLUX.2-klein-9B
| Config | VRAM |
|---|---|
| bf16 no fp8 | ~35 GB |
| fp8+fp8_scaled | ~22 GB |
| fp8+fp8_scaled + blocks_to_swap=16 | ~16 GB |
| fp8+fp8_scaled + blocks_to_swap=16 + cpu_offload | ~12 GB |

### OOM? Increase `BLOCKS_TO_SWAP` in Section 6 (max = 16 for this model).


In [ ]:
import os
REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

if TEXT_ENCODER_PATH is None:
    raise RuntimeError("TEXT_ENCODER_PATH not set — re-run Section 7.")

# Memory flags
mem_flags = []
if USE_FP8:
    mem_flags += ["--fp8_base", "--fp8_scaled"]   # FLUX.2 needs BOTH for fp8
if GRADIENT_CHECKPOINTING:
    mem_flags.append("--gradient_checkpointing")
if GRADIENT_CHECKPOINTING_CPU_OFFLOAD:
    mem_flags.append("--gradient_checkpointing_cpu_offload")
if BLOCKS_TO_SWAP > 0:
    mem_flags.append(f"--blocks_to_swap {BLOCKS_TO_SWAP}")
if VAE_DTYPE != "float32":
    mem_flags.append(f"--vae_dtype {VAE_DTYPE}")

# TensorBoard
log_flags = ""
if USE_TENSORBOARD:
    log_dir = f"{OUTPUT_DIR}/logs"
    os.makedirs(log_dir, exist_ok=True)
    log_flags = f"--log_with tensorboard --logging_dir {log_dir}"

cmd = (
    f"accelerate launch"
    f" --num_cpu_threads_per_process 1"
    f" --mixed_precision {MIXED_PRECISION}"
    f" src/musubi_tuner/flux_2_train_network.py"
    f" --model_version {MODEL_VERSION}"
    f" --dit {DIT_PATH}"
    f" --vae {AE_PATH}"
    f" --text_encoder {TEXT_ENCODER_PATH}"
    f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
    f" --sdpa"
    f" --mixed_precision {MIXED_PRECISION}"
    f" --timestep_sampling flux2_shift"
    f" --weighting_scheme none"
    f" {' '.join(mem_flags)}"
    f" --optimizer_type {OPTIMIZER_TYPE}"
    f" --learning_rate {LEARNING_RATE}"
    f" --max_data_loader_n_workers 2"
    f" --persistent_data_loader_workers"
    f" --network_module networks.lora_flux_2"
    f" --network_dim {NETWORK_DIM}"
    f" --network_alpha {NETWORK_ALPHA}"
    f" --max_train_epochs {MAX_TRAIN_EPOCHS}"
    f" --save_every_n_epochs {SAVE_EVERY_N_EPOCHS}"
    f" --seed {SEED}"
    f" --output_dir {OUTPUT_DIR}"
    f" --output_name {LORA_NAME}"
    f" {log_flags}"
)

print("Training command:")
print(cmd)
print("\n" + "="*60)
!{cmd}


## Section 13 — (Optional) Test Inference

Generate a test image using your trained LoRA.

FLUX.2-klein-9B requires a **control image** (reference image) for generation.  
Edit `INFER_PROMPT` and `CONTROL_IMAGE_PATH` below.


In [ ]:
import os, glob
REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

# ---- Edit these ----
INFER_PROMPT        = "a photo of sks person smiling in a park, cinematic lighting"
CONTROL_IMAGE_PATH  = ""   # path to a reference image; required for FLUX.2
INFER_IMAGE_SIZE    = [1024, 1024]   # [H, W] (note: height first for this script)
INFER_STEPS         = 4              # 4 for klein (distilled), 50 for klein-base
INFER_GUIDANCE      = 1.0           # 1.0 for klein, 4.0 for klein-base
INFER_SEED          = 42
INFER_OUTPUT_DIR    = f"{DRIVE_ROOT}/inference_output"
os.makedirs(INFER_OUTPUT_DIR, exist_ok=True)

if not CONTROL_IMAGE_PATH:
    print("⚠️  CONTROL_IMAGE_PATH is empty.")
    print("   Set it to a reference image path and re-run.")
else:
    # Auto-pick latest LoRA checkpoint
    lora_files = sorted(
        glob.glob(f"{OUTPUT_DIR}/{LORA_NAME}*.safetensors"),
        key=os.path.getmtime
    )
    if not lora_files:
        print("⚠️  No LoRA checkpoints found. Train first.")
    else:
        LORA_PATH = lora_files[-1]
        print(f"Using LoRA: {LORA_PATH}")

        fp8_flag = "--fp8_scaled" if USE_FP8 else ""

        cmd = (
            f"python src/musubi_tuner/flux_2_generate_image.py"
            f" --model_version {MODEL_VERSION}"
            f" --dit {DIT_PATH}"
            f" --vae {AE_PATH}"
            f" --text_encoder {TEXT_ENCODER_PATH}"
            f" --control_image_path \"{CONTROL_IMAGE_PATH}\""
            f" --prompt \"{INFER_PROMPT}\""
            f" --image_size {INFER_IMAGE_SIZE[0]} {INFER_IMAGE_SIZE[1]}"
            f" --infer_steps {INFER_STEPS}"
            f" --guidance_scale {INFER_GUIDANCE}"
            f" {fp8_flag}"
            f" --save_path {INFER_OUTPUT_DIR}"
            f" --output_type images"
            f" --seed {INFER_SEED}"
            f" --lora_multiplier 1.0"
            f" --lora_weight {LORA_PATH}"
        )

        print("Inference command:")
        print(cmd)
        print()
        !{cmd}

        # Display output image
        out_imgs = sorted(glob.glob(f"{INFER_OUTPUT_DIR}/*.png"))
        if out_imgs:
            from IPython.display import Image, display
            display(Image(out_imgs[-1], width=512))


## Section 14 — List LoRA Checkpoints


In [ ]:
import os, glob

lora_files = sorted(
    glob.glob(f"{OUTPUT_DIR}/*.safetensors"),
    key=os.path.getmtime
)

if not lora_files:
    print(f"No LoRA files found in {OUTPUT_DIR}")
else:
    print(f"LoRA checkpoints in {OUTPUT_DIR}:")
    for f in lora_files:
        size_mb = os.path.getsize(f) / 1024**2
        print(f"  {os.path.basename(f):50s}  {size_mb:7.1f} MB")
    print(f"\nLatest: {os.path.basename(lora_files[-1])}")
    print("Files are on Google Drive — download from there or uncomment below.")

# Uncomment to download latest checkpoint directly to browser:
# from google.colab import files
# if lora_files:
#     files.download(lora_files[-1])


## Tips & Troubleshooting

### Out of Memory (OOM)
In Section 6, progressively enable more memory options:
1. `USE_FP8 = True` (saves ~13 GB — always enable this first)
2. `BLOCKS_TO_SWAP = 8` → `16` (max for klein-9b)
3. `GRADIENT_CHECKPOINTING_CPU_OFFLOAD = True`
4. Reduce `RESOLUTION` to `[768, 768]` or `[512, 512]`
5. Reduce `NETWORK_DIM` to 16

### Should I use klein-9B or klein-base-9B for training?
The musubi-tuner docs recommend **klein-base-9B** for training because klein-9B is a distilled model.  
To switch: change `MODEL_VERSION = "klein-base-9b"` and `DIT_FILE = "flux2-klein-base-9b.safetensors"` in Section 6,
then download from `black-forest-labs/FLUX.2-klein-base-9B` in Section 7.

### Colab disconnects mid-training
All checkpoints are on Google Drive. Re-run Sections 3, 4, 5, 6 to restore the environment,
then go straight to Section 12 to resume training from epoch 1 (musubi-tuner doesn't auto-resume yet).

### Convert LoRA for ComfyUI
```bash
python src/musubi_tuner/convert_lora.py \
  --input /path/to/musubi_lora.safetensors \
  --output /path/to/comfyui_lora.safetensors \
  --target other
```

### Links
- [musubi-tuner GitHub](https://github.com/kohya-ss/musubi-tuner)
- [FLUX.2 training docs](https://github.com/kohya-ss/musubi-tuner/blob/main/docs/flux_2.md)
- [hf CLI reference](https://huggingface.co/docs/huggingface_hub/en/guides/cli)
- [FLUX.2-klein-9B model page](https://huggingface.co/black-forest-labs/FLUX.2-klein-9B)
